## MODELING (MEDIAN AND FREQUENCY IMPUTATION VERSION)


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import flaml
import sys
import os
import xgboost as xgb
from flaml import AutoML

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression

#adding the parent directory to the system path to import the custom module
sys.path.append(os.path.abspath(os.path.join('..')))

from src.utils.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [3]:
file_path = r'C:\unibo-dtm-ml-2526-cervical-cancer-predictor\data\data_after_imputation\median_and_freq_imputed.csv'
df = pd.read_csv(file_path)

# repeat the data profiling pipeline for the newly cleaned data
df = pd.read_csv(file_path)

print("\nDataset Info: \n")
print(df.info())

#check whether everything went smoothly at the data cleaning stage
print("\nMissing Values: \n")
print(df.isnull().sum()) 


print("\nDescriptive Statistics:")
print(df.describe(include='all'))



Dataset Info: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 23 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Age                              835 non-null    float64
 1   Number of sexual partners        835 non-null    float64
 2   First sexual intercourse         835 non-null    float64
 3   Num of pregnancies               835 non-null    float64
 4   Smokes (years)                   835 non-null    float64
 5   Smokes (packs/year)              835 non-null    float64
 6   Hormonal Contraceptives (years)  835 non-null    float64
 7   IUD (years)                      835 non-null    float64
 8   STDs (number)                    835 non-null    float64
 9   STDs: Viral group                835 non-null    float64
 10  STDs: Bacterial group            835 non-null    float64
 11  STDs:condylomatosis              835 non-null    float64
 12  STDs:

In [4]:
targets = ['Biopsy', 'Hinselmann', 'Schiller', 'Citology']
y = df[targets]
X = df.drop(columns=targets)

"""use msss (MultilabelStratifiedShuffleSplit) to split the data into train and test sets, 
ensuring that the distribution of the target variables is preserved in both sets.
This is particularly important in multilabel classification problems, where each instance can belong to multiple classes simultaneously.
Furthermore, this form of data splitting is needed for such an imbalanced dataset, where some classes may be underrepresented. 
"""

msss = MultilabelStratifiedShuffleSplit(n_splits=5, test_size=0.2,random_state=42)
msss.get_n_splits(X, y)

#showing how the data is iteratively split into train and test sets, while preserving the distribution of the target variables
for train_index, test_index in msss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")


print(f"Training samples: {X_train.shape[0]} ({len(y_train[y_train.sum(axis=1) > 0])} with at least one positive target)")
print(f"Testing samples:  {X_test.shape[0]} ({len(y_test[y_test.sum(axis=1) > 0])} with at least one positive target)")

X_train: (668, 19)
X_test: (167, 19)
y_train: (668, 4)
y_test: (167, 4)
Training samples: 668 (84 with at least one positive target)
Testing samples:  167 (17 with at least one positive target)


### LOGISTIC REGRESSION

Specifically following the One-vs-Rest strategy, which involves training binary classifiers for each class against all other classes combined.

In [5]:
from sklearn.multioutput import MultiOutputClassifier

ovr_clf = MultiOutputClassifier(LogisticRegression(class_weight='balanced', random_state=42),n_jobs=-1)
count = 0

for train_index, test_index in msss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    ovr_clf.fit(X_train, y_train)
    y_pred = ovr_clf.predict(X_test)
    count += 1
    
    print(f"--- Baseline Multi-Label Report for fold {count} ---")
    print(classification_report(y_test, y_pred, target_names=targets, zero_division=0))

--- Baseline Multi-Label Report for fold 1 ---
              precision    recall  f1-score   support

      Biopsy       0.16      0.27      0.20        11
  Hinselmann       0.00      0.00      0.00         7
    Schiller       0.20      0.33      0.25        15
    Citology       0.08      0.33      0.12         9

   micro avg       0.10      0.26      0.15        42
   macro avg       0.11      0.23      0.14        42
weighted avg       0.13      0.26      0.17        42
 samples avg       0.03      0.05      0.03        42

--- Baseline Multi-Label Report for fold 2 ---
              precision    recall  f1-score   support

      Biopsy       0.17      0.55      0.26        11
  Hinselmann       0.03      0.14      0.04         7
    Schiller       0.16      0.47      0.23        15
    Citology       0.07      0.33      0.12         9

   micro avg       0.10      0.40      0.17        42
   macro avg       0.10      0.37      0.16        42
weighted avg       0.12      0.40    

It is possible to note that there's very high values for recall and very low for precision. This is right: this dataset is imbalanced and, most importantly, is a medical diagnosis dataset. The meaning behind these score is the following: the linear model opts for flagging one patient as positive even when there's only a slight doubt. It prioritizes correctly flagging all TPs, at the cost of producing, at the same time, many more FPs labels. This is all caused by one linear model attempting to predict somrething that is extremely non-linear, as a diagnostic test result sees too many variables (risk factors) being interactive and having non-linear relationships with each other. This is why I will now try to have a go with XGBOOST and see the difference. 

### XGBOOST


Tree-based technique relying on boosting. This means it will be able to better handle the imbalanced dataset. Samples that are wrongly classified will be increasingly more important the more the classifier gets them wrong over multiple iterations. 

In [6]:
xgb_estimator = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=13,  #forcing the tree to prioritize the minority class
    max_depth=1,          #kept it shallow to prevent overfitting on the small dataset
    learning_rate=0.01,   #slow learning rate for better stability
    n_estimators=500,     #more trees to allow the model to learn complex patterns, but with early stopping to prevent overfitting
    reg_lambda = 100,     #strong regularization to prevent overfitting on the small dataset
    random_state=42
    )

multilabel_model = MultiOutputClassifier(xgb_estimator, n_jobs=-1)

#list to store the classification reports for each fold in the form of dictionaries
all_reports = []

for train_index, test_index in msss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    multilabel_model.fit(X_train, y_train)
    y_pred = multilabel_model.predict(X_test)
    
    report_dict = classification_report(y_test, y_pred, target_names=targets, zero_division=0, output_dict=True)
    all_reports.append(report_dict)

#calculate the average precision, recall, and f1-score for each target across all folds
final_results = {}
for target in targets + ['macro avg']:
    final_results[target] = {
        'precision': np.mean([r[target]['precision'] for r in all_reports]),
        'recall': np.mean([r[target]['recall'] for r in all_reports]),
        'f1-score': np.mean([r[target]['f1-score'] for r in all_reports])
    }

#display the final results in a DataFrame for better readability

print("=== FINAL AVERAGE PERFORMANCE (5-FOLDS) ===")
print(pd.DataFrame(final_results).T.round(3))

=== FINAL AVERAGE PERFORMANCE (5-FOLDS) ===
            precision  recall  f1-score
Biopsy          0.145   0.473     0.221
Hinselmann      0.056   0.086     0.063
Schiller        0.170   0.560     0.261
Citology        0.067   0.111     0.083
macro avg       0.110   0.307     0.157


In [7]:


# 1. Initialize a list to store the dictionaries from each fold
all_reports = []

for train_index, test_index in msss.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Use your winning XGBoost settings
    multilabel_model.fit(X_train, y_train)
    y_pred = multilabel_model.predict(X_test)
    
    # Capture the report as a dictionary
    report_dict = classification_report(y_test, y_pred, target_names=targets, 
                                        output_dict=True, zero_division=0)
    all_reports.append(report_dict)

# 2. Average the results for each target and each metric
final_results = {}
for target in targets + ['macro avg']:
    final_results[target] = {
        'precision': np.mean([r[target]['precision'] for r in all_reports]),
        'recall': np.mean([r[target]['recall'] for r in all_reports]),
        'f1-score': np.mean([r[target]['f1-score'] for r in all_reports])
    }

# 3. Display as a clean DataFrame
final_df = pd.DataFrame(final_results).T
print("=== FINAL AVERAGE PERFORMANCE (5-FOLDS) ===")
print(final_df.round(3))

=== FINAL AVERAGE PERFORMANCE (5-FOLDS) ===
            precision  recall  f1-score
Biopsy          0.145   0.473     0.221
Hinselmann      0.056   0.086     0.063
Schiller        0.170   0.560     0.261
Citology        0.067   0.111     0.083
macro avg       0.110   0.307     0.157


### RANDOM FOREST

### SVM

### MULTI-LABEL KNN